# Lecture 02 Exercise: Learn an Answer Format with LoRA and Compare Before and After Training

Using a fictional course FAQ, briefly train a small instruction-tuned model with LoRA and compare **what changes and what stays the same** before and after training. Evaluation separates **format compliance rate** (whether it follows the JSON format with `answer` and `needs_review`) and **content accuracy**.

**What to Check in This Exercise**

1. A small model may fail to follow format instructions even when they are in the prompt (section 3, before training).
2. In the original Japanese exercise, it followed the format almost consistently after a few minutes of LoRA training; check whether your run does too (section 5, after training).
3. A correct format does not guarantee sound judgment when asked about things absent from the FAQ (whether to request confirmation or make an assertion; sections 5 and 6).
4. Advanced: in the original Japanese exercise, loading the same model with 4-bit quantization changed outputs even without training; compare your own results (section 8).

**Workflow**

| Section | Contents | Estimated Time (T4, Original Japanese Exercise) |
| --- | --- | --- |
| 0–3 | Environment, settings, data generation, model loading, before-training evaluation | About 5 minutes |
| 4–7 | Training, after-training evaluation, knowledge questions, comparison table and submission | About 10 minutes (including 4–5 minutes of training) |
| 8 (optional) | Load in 4-bit and compare | About 2 minutes |

Running 0–3 before class to check the environment will help avoid delays during class (optional). Colab loses variables when the session disconnects; if that happens, start again from 0 on the day.

**Requirements**: Google Colab (T4 GPU, free tier). Model: `Qwen/Qwen3.5-0.8B` (Apache-2.0). This notebook generates fictional data, with no real deadlines or URLs.

**How to Read**: Each section explains "What to Do", "What to Look For", and "Troubleshooting". Run cells in order from the top. Skipping cells causes missing-variable errors in later cells.

**Translation note**: The timing, loss, and output observations quoted here come from the original Japanese exercise. Results may differ with the translated FAQ and questions; these observations have not been remeasured for this translation.

Version: 1.1 (2026-10-18)

## 0. Check the Environment and Install Libraries

**What to Do**: Check that a GPU is allocated and install libraries pinned to verified versions.

**What to Look For**

- The first cell displays `Tesla T4`.
- The second cell ends with a list of versions (such as `transformers: 5.18.0`). Installation takes about 1 minute.

**Troubleshooting**

- "No GPU found": select **Runtime → Change runtime type → T4 GPU** and run the cell again. If demand prevents allocation, try again later.
- Red pip `ERROR: pip's dependency resolver ...` messages (about `cudf` or `pyarrow`) warn of conflicts with other packages preinstalled in Colab. They do not affect this exercise, so continue.

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU runtimes do not have nvidia-smi
    gpu_info = ""
print(gpu_info or "No GPU found. Select Runtime → Change runtime type → T4 GPU.")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Versions verified on a Colab T4 (Python 3.13, torch 2.11.0+cu130). Use the torch bundled with Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 requires transformers 5.x
    "peft": "0.21.1",           # LoRA implementation
    "trl": "1.14.2",            # Training loop (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # The bundled torchao (0.10) is incompatible with peft 0.21 and causes ImportError during training. Remove it since it is unused
%pip install -q {spec}

# Dependency warnings about cudf-cu13 and pyarrow appear as ERROR, but can be ignored because cudf is unused
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. Settings

**What to Do**: Collect the model name, number of training epochs (passes through the data), and other settings in a single dictionary, `CONFIG`. This dictionary is recorded as is in the submission file in section 7 so you can later identify "the settings used for each experiment".

- Fixing `revision` ensures the same weights are used whenever you run the notebook. This keeps conditions consistent for before/after and cross-person comparisons.
- `num_train_epochs: 20` means 20 passes through the training data (14 examples). With 2 steps per pass, this totals 40 steps. The original Japanese run took 4–5 minutes on a T4; timing may differ with the translated data.
- `gen: {"do_sample": False}` uses greedy generation. This fixes generation conditions before and after training and removes the effect of sampling variation.

Setting `USE_DRIVE` to `True` mounts Google Drive and saves the trained adapter and result files to Drive as well. Use this to retain results if the session disconnects. Leave it `False` if you will finish in one session.

**What to Look For**: The settings are displayed as JSON.

In [1]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # Small instruction-tuned model (Apache-2.0). Fits on a T4 without quantization
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # Pinned to use the same weights on every run
    "seed": 42,
    "num_train_epochs": 5,  # Number of passes over training data (14 examples). 2 steps per pass gives 40 steps (4–5 minutes on a T4 in the original Japanese run)
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA matrix dimension (smaller means fewer trainable parameters)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 128,
    "gen": {"do_sample": False},   # Fix evaluation to greedy generation (consistent before/after conditions)
    "work_dir": "/content/qlora_lab",                          # Result file location (visible in the file list on the left)
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # Save location when USE_DRIVE=True
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # Set True to mount Google Drive and save adapters and outputs
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 5,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 128,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import json
raw = [json.loads(l) for l in open("/content/drive/MyDrive/glossary_sft.jsonl")]
print(len(raw))
print(raw[0]["messages"][0])

116
{'role': 'user', 'content': 'What is AI engineering?'}


In [6]:
terms = sorted({r["messages"][0]["content"].replace("What is ","").replace("Briefly explain ","").rstrip("?.") for r in raw})
print(len(terms)); print(terms)

58
['3 homework levels', 'ADK (Agent Development Kit)', 'AI agent', 'AI engineering', 'API', 'API key', 'Agent loop', 'Branch', 'Code completion', 'Codespaces secret', 'Coding agent', 'Context window', 'Default model', 'Environment variable', 'Epoch', 'Evaluation and improvement cycle', 'External tool', 'Fine-tuning', 'GitHub Codespaces', 'GitHub Copilot', 'Guardrail', 'Hallucination', 'Harness', 'Hugging Face Hub', 'Human-in-the-loop', 'LLM (Large Language Model)', 'Learning rate', 'LoRA', 'MCP (Model Context Protocol)', 'MCP client', 'MCP server', 'Overfitting', 'PEFT', 'Pi', 'Pretraining', 'Prompt', 'Pull Request (PR)', 'Pydantic AI', 'QLoRA', 'Quantization', 'Rank (LoRA r)', 'Rate limit', 'Structured output', 'Supervised Fine-Tuning (SFT)', 'System prompt', 'TestModel', 'Token', 'Tool (Tool use / Function calling)', 'Tool description', 'Tool namespace', 'Transport (stdio / Streamable HTTP)', 'answer.md', 'devcontainer', 'fetch server', 'fork', 'lecNN-start tag', 'mcpServers configu

## 2. Generate a Fictional FAQ and Training Data

**What to Do**: Create training (train), validation (valid), and evaluation (test) data from a fictional FAQ with 5 items.

Model input is "FAQ + question"; the desired output is one JSON line: `{"answer": <string>, "needs_review": <boolean>}`. In the example filename `student_id_assignment_number.pdf`, replace `student_id` with your student ID and `assignment_number` with the assignment number.

- For questions with evidence in the FAQ, answer with the FAQ text and set `needs_review: false`.
- For questions absent from the FAQ, answer "This FAQ does not contain that information. Confirmation from the course organizers is needed." and set `needs_review: true`.

**Data Splits**

- Paraphrases asking about the same FAQ item are assigned to train / valid / test **by template**, so text seen during training does not appear unchanged in test.
- test has 21 questions (15 with evidence, 6 without). Additional test-only questions ensure this count.
- Expected answers (the `needs_review` value and required answer keywords) are held separately in `gold` and are not passed to the model.
- train has 14 examples (10 with evidence, 4 without), and valid has 7. With only 5 FAQ items, training data is limited, so the same examples are trained on repeatedly. The number of repetitions is controlled by the number of epochs (`num_train_epochs`) in section 4, rather than duplicating data. The standard approach is to put only actual examples in data files and control training volume through training settings.

**What to Look For**: `train=14 valid=7 test=21 gold=21` and one training example (question and desired output) are displayed. Duplicate questions anywhere across train / valid / test cause an error here.

In [ ]:
# FAQ = {
#     "deadline": "The assignment deadline is September 30 at 23:59 (Japan time).",
#     "destination": "Submit through the 'Assignments' section of the learning portal.",
#     "format": "Submit one PDF file named student_id_assignment_number.pdf.",
#     "office_hours": "Questions are welcome during office hours every Tuesday from 17:00 to 18:00.",
#     "late": "Late submissions are accepted within 48 hours of the deadline, with a grade penalty.",
# }
# FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# # With evidence: (FAQ key, question templates, required keywords in the expected answer)
# GROUNDED = {
#     "deadline": (["When is the assignment deadline?", "By when should I submit?", "Tell me the deadline", "What is the submission deadline?"], ["September 30", "23:59"]),
#     "destination": (["Where should I submit?", "Where do I upload it?", "Where can I submit the assignment?", "Where should I submit my work?"], ["learning portal", "Assignments"]),
#     "format": (["Tell me the submission format", "Are there any file naming rules?", "Which file format should I submit?", "Can I submit a format other than PDF?"], ["PDF"]),
#     "office_hours": (["When can I ask questions?", "When are office hours?", "Is there a time to ask the instructor questions?", "What times are available for consultation?"], ["Tuesday", "17:00"]),
#     "late": (["What happens if I submit late?", "Are submissions accepted after the deadline?", "What are the late submission rules?", "Tell me how submissions after the deadline are handled"], ["48 hours", "penalty"]),
# }
# # Without evidence: absent from the FAQ, so needs_review=true is correct
# UNGROUNDED = [
#     "Can I resubmit?", "Tell me how missed lectures are handled", "Can I see other participants' scores?",
#     "When will grades be released?", "Can the submission deadline be extended?", "Are group submissions allowed?",
#     "How are points allocated for assignments?", "Will lecture recordings be made available?",
# ]

# # Additional test-only questions (excluded from train / valid)
# TEST_EXTRA_GROUNDED = {
#     "deadline": ["Tell me the submission deadline date and time", "On what date and at what time is the assignment due?"],
#     "destination": ["Where do I hand in the assignment?", "Tell me where to send my submission"],
#     "format": ["What format should the submitted file use?", "What file format and how many files should I submit?"],
#     "office_hours": ["Tell me the day and time of office hours", "On which day and from what time can I ask questions?"],
#     "late": ["What happens if I miss the submission deadline?", "Will an assignment submitted after the deadline be accepted?"],
# }
# TEST_EXTRA_UNGROUNDED = [
#     "Is there a character count limit for assignments?", "Will I receive feedback on my submitted assignment?",
#     "Is a particular citation style required?", "Do office hours require an appointment?",
# ]
# UNGROUNDED_ANSWER = "This FAQ does not contain that information. Confirmation from the course organizers is needed."

# def make_example(question: str, answer: str, needs_review: bool) -> dict:
#     target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
#     return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

# def split_templates(templates):
#     # Fix the first 2 templates to train, the third to valid, and the fourth to test (keep paraphrases separate)
#     return templates[:2], templates[2:3], templates[3:4]

# train, valid, test, gold = [], [], [], {}
# for key, (templates, must_include) in GROUNDED.items():
#     tr, va, te = split_templates(templates)
#     for q in tr: train.append(make_example(q, FAQ[key], False))
#     for q in va: valid.append(make_example(q, FAQ[key], False))
#     for q in te + TEST_EXTRA_GROUNDED[key]:
#         test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
# for i, q in enumerate(UNGROUNDED):
#     ex = make_example(q, UNGROUNDED_ANSWER, True)
#     bucket = [train, train, valid, test][i % 4]   # Assign each group of 4 questions without evidence as train 2 / valid 1 / test 1
#     bucket.append(ex)
#     if bucket is test: gold[q] = {"needs_review": True, "must_include": ["Confirmation"]}
# for q in TEST_EXTRA_UNGROUNDED:
#     test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["Confirmation"]}

# # train has 14 examples (10 with evidence, 4 without). Set repetitions with num_train_epochs in section 4 instead of duplicating data
# all_questions = [r["question"] for r in train + valid + test]
# assert len(set(all_questions)) == len(all_questions), "Questions are duplicated across train / valid / test. Review the templates"
# print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
# print("One training example | Question:", train[0]["question"], "| Desired output:", train[0]["target"])
# _ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
# _ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
# _ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

In [8]:
import json, re

raw = [json.loads(l) for l in open("/content/drive/MyDrive/glossary_sft.jsonl")]

def term_of(q):
    m = re.fullmatch(r"(?:What is|Briefly explain) (.+)[?.]", q)
    return m.group(1) if m else None

# FAQ equivalent: loaded from the file
GLOSSARY = {term_of(r["messages"][0]["content"]): r["messages"][1]["content"]
            for r in raw if r["messages"][0]["content"].startswith("What is ")}
print(len(raw), len(GLOSSARY))      # expect 116 and 58

# GROUNDED equivalent: three kinds of glossary terms
TEST_TERMS    = ["LoRA", "Hallucination", "Token", "Context window"]                   # seen in training (exact + reworded questions)
HOLDOUT_TERMS = ["3 homework levels", "Structured output", "Learning rate", "Guardrail"]  # in the glossary, removed from training
VALID_TERMS   = ["Epoch", "Prompt", "Branch", "Harness", "Quantization", "fork"]       # validation only
INCLUDE_EXACT = True                                                                   # also test the exact training wording

TRAIN_TEMPLATES = ["What is {}?", "Briefly explain {}."]                                # already in the file
VALID_TEMPLATE  = "Define {}."
TEST_TEMPLATES  = ["Can you tell me what {} means?",
                   "Explain {} in the context of this course.",
                   "Give a brief definition of {}.",
                   "What does the term {} refer to?"]                                   # test only
# Natural wording for a term whose name reads awkwardly in the templates (the second one does not contain the term name)
EXTRA_HELDOUT = {"3 homework levels": ["What are the 3 homework levels?",
                                        "What are the three levels of homework in this course?"]}

# UNGROUNDED equivalent: terms an LLM may know, but that are not glossary entries
ABSENT_TERMS = ["RAG", "Embedding", "Vector database", "DSPy", "LangGraph", "Agentic RAG", "ReAct"]

# ---- sanity checks on the term lists ----
for name, terms in [("test", TEST_TERMS), ("holdout", HOLDOUT_TERMS), ("valid", VALID_TERMS)]:
    assert all(t in GLOSSARY for t in terms), (name, [t for t in terms if t not in GLOSSARY])
assert not set(ABSENT_TERMS) & set(GLOSSARY), set(ABSENT_TERMS) & set(GLOSSARY)
assert len(set(TEST_TERMS + HOLDOUT_TERMS + VALID_TERMS)) == len(TEST_TERMS + HOLDOUT_TERMS + VALID_TERMS), "a term is in two lists"
assert all(t in HOLDOUT_TERMS for t in EXTRA_HELDOUT)

def to_record(q, a):
    return {"messages": [{"role": "user", "content": q}, {"role": "assistant", "content": a}]}

# train / valid (messages format, passed straight to Dataset.from_list)
train_records = [r for r in raw if term_of(r["messages"][0]["content"]) not in HOLDOUT_TERMS]
valid_records = [to_record(VALID_TEMPLATE.format(t), GLOSSARY[t]) for t in VALID_TERMS]

# test + gold
test, gold = [], {}
def add(q, group, term):
    assert q not in gold, f"duplicate question: {q}"
    test.append({"question": q})
    gold[q] = {"group": group, "term": term,
               "expect": "answer" if term in GLOSSARY else "unknown",
               "reference": GLOSSARY.get(term)}

for t in TEST_TERMS:
    if INCLUDE_EXACT: add(TRAIN_TEMPLATES[0].format(t), "exact", t)
    for tmpl in TEST_TEMPLATES: add(tmpl.format(t), "paraphrase", t)
for t in HOLDOUT_TERMS:
    for tmpl in [TRAIN_TEMPLATES[0]] + TEST_TEMPLATES: add(tmpl.format(t), "heldout", t)
for t, qs in EXTRA_HELDOUT.items():
    for q in qs: add(q, "heldout", t)
for t in ABSENT_TERMS:
    add(TRAIN_TEMPLATES[0].format(t), "absent", t)

# ---- leakage checks ----
train_q = {r["messages"][0]["content"] for r in train_records}
for q, g in gold.items():
    if g["group"] == "exact": assert q in train_q, q            # exact wording is in training on purpose
    else: assert q not in train_q, f"leak: {q}"
assert len(gold) == len(test)

# ---- how often does each held-out / absent term already appear inside other training answers? ----
train_text = " ".join(r["messages"][1]["content"].lower() for r in train_records)
print("mentions in training answers:", {t: len(re.findall(re.escape(t.lower()), train_text)) for t in HOLDOUT_TERMS + ABSENT_TERMS})

print(f"train={len(train_records)} valid={len(valid_records)} test={len(test)}",
      {g: sum(v["group"] == g for v in gold.values()) for g in ["exact", "paraphrase", "heldout", "absent"]})

116 58
mentions in training answers: {'3 homework levels': 0, 'Structured output': 0, 'Learning rate': 0, 'Guardrail': 0, 'RAG': 6, 'Embedding': 0, 'Vector database': 0, 'DSPy': 0, 'LangGraph': 0, 'Agentic RAG': 0, 'ReAct': 0}
train=108 valid=6 test=49 {'exact': 4, 'paraphrase': 16, 'heldout': 22, 'absent': 7}


## 3. Load the Model and Evaluate Before Training

**What to Do**: Load the base model without quantization (32-bit), have it answer the 21 test questions, and score the results.

The prompt (`SYSTEM`) says "Always return `answer` and `needs_review` in one JSON line". The **format has already been instructed**. Before training, check whether the model follows this instruction.

**Scoring** (the same functions are used in section 5)

- **Format**: The output parses as JSON, has exactly two keys (`answer` and `needs_review`), and uses string and boolean types respectively.
- **Content**: `needs_review` matches the expected answer, and `answer` contains the required keywords (for example, "September 30" and "23:59").

Content scoring requires a matching `needs_review` value. Outputs with a missing key therefore receive `Fail` for content even if the wording is correct. Check the wording yourself in the table. The format-failure codes mean: `json_error` = JSON parsing failed; `not_object` = the JSON value is not an object; `extra_keys` = unexpected keys are present; `missing_key` = a required key is absent; `wrong_type` = an answer or review flag has the wrong type.

`Qwen/Qwen3.5-0.8B` also supports images, but loading it with `AutoModelForCausalLM` uses only the text component.

**What to Look For**

- First cell: a message such as `Weight memory (GB): 3.01`. Downloading the model (about 1.75GB) takes about 1 minute.
- Second cell: a one-line summary (`format_rate`, `content_rate`, etc.) and a table of outputs for 21 questions. Evaluation takes about 1 minute. In the original Japanese run, many before-training outputs were missing `needs_review`; check which keys are missing in your run.

**Harmless Warnings**

- `You are sending unauthenticated requests to the HF Hub`: this means the model is being downloaded without signing in to Hugging Face. Public models can be downloaded without signing in. Signing in is needed in the following cases, regardless of model size:
  - Using a model that requires agreement to terms (a gated model, such as Llama or Gemma) or a private repository.
  - Uploading your own models or data to the Hub.
  - Sending many requests in a short time and hitting a limit. Without signing in, the limit is 3,000 file retrievals per IP address per 5 minutes (published figure as of September 2025; [Hub rate limits](https://huggingface.co/docs/hub/rate-limits)). Signing in raises it.

  This notebook's model is public and requires no agreement. Loading it makes at most a few dozen requests, well below the limit. If `429 Too Many Requests` does occur, the library reads the waiting time and retries automatically. If signing in becomes necessary, register a Hugging Face token named `HF_TOKEN` using the key icon (Secrets) on Colab's left sidebar and allow this notebook to access it; it will be used automatically.
- `... is falling back to its reference PyTorch implementation ...`: a fast implementation for some of this model's layers (linear attention) is not installed, so a slower but correct implementation is used. About 4 lines appear.

**Read Before Continuing**: Check the table for "which keys are missing" and "whether the wording agrees with the FAQ". In section 5, compare this table alongside outputs after training. Original outputs are also saved in `/content/qlora_lab/eval_before.json` (open it from the file list on the left).

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# No quantization. Keep weights in 32-bit (specify dtype because omitting it loads the distributed precision)
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| Weight memory (GB):", BASE_MEMORY_GB)

# The prompt instructs the format. Before training, check whether the model follows it
# SYSTEM = "You answer course FAQ questions. Use only the provided FAQ as evidence and always answer in one JSON line. Format: {\"answer\": string, \"needs_review\": boolean}. For information absent from the FAQ, write 'Confirmation is needed' in answer and set needs_review to true."

def build_messages(example):          # closed-book: just the question, matching the training format
    return [{"role": "user", "content": example["question"]}]

def generate(model, example):
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | Weight memory (GB): 3.01


In [10]:
import re, time, json
import pandas as pd
from difflib import SequenceMatcher
pd.set_option("display.max_colwidth", None)   # Display original outputs without truncation

THRESH = 0.8   # similarity >= 0.8 counts as "matches the glossary definition"
REFUSAL = re.compile(r"not (?:in|listed|found|provided|mentioned)|no (?:information|such)|don't (?:know|have)|do not (?:know|have)|cannot|can't|unknown|confirm|unable|more (?:context|details)|could you (?:please )?(?:provide|share)", re.I)
REFUSED = ["possible refusal", "declined"]      # labels that count as "declined to answer"

def check_content(output: str, gold_entry):
    '''Terms in the glossary: similarity to the definition. Terms not in the glossary: keyword guess at whether it declined (read these yourself)'''
    if gold_entry["expect"] == "answer":
        sim = round(SequenceMatcher(None, output, gold_entry["reference"], autojunk=False).ratio(), 2)
        return sim, ("match" if sim >= THRESH else "mismatch")
    return None, ("possible refusal" if REFUSAL.search(output) else "asserted?")

def grp(df):
    '''Per-group table for questions that have a glossary definition'''
    a = df[df.group != "absent"]
    return a.groupby("group").agg(n=("label", "size"), mean_similarity=("similarity", "mean"),
                                  match_rate=("label", lambda s: (s == "match").mean())).round(2)

def absent_stats(df):
    '''How many absent-term questions were declined (keyword guess)'''
    a = df[df.group == "absent"]
    n = int(a.label.isin(REFUSED).sum())
    return {"refused": n, "of": int(len(a)), "rate": round(n / len(a), 2)}

def headline(df):
    '''One number: glossary questions count when they match, absent questions count when declined'''
    ok = ((df.group != "absent") & (df.label == "match")) | ((df.group == "absent") & df.label.isin(REFUSED))
    return {"micro": round(float(ok.mean()), 2), "macro": round(float(ok.groupby(df.group).mean().mean()), 2)}

def match_at(df, th):
    '''Match rate per group at a different threshold (sensitivity check)'''
    a = df[df.group != "absent"]
    return (a.similarity >= th).groupby(a.group).mean().round(2)

def evaluate(model, label: str):
    '''Generate and score every test question. Save original outputs in glossary_<label>.json'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        sim, lab = check_content(raw, gold[ex["question"]])
        rows.append({"group": gold[ex["question"]]["group"], "question": ex["question"],
                     "output": raw, "similarity": sim, "label": lab})
    df = pd.DataFrame(rows)
    seconds = round(time.time() - t0, 1)
    df.to_json(WORK / f"glossary_{label}.json", orient="records", force_ascii=False, indent=2)
    print(f"[{label}] {seconds}s")
    print(grp(df))
    print("absent refusal:", absent_stats(df), "| labels:", df[df.group == "absent"]["label"].value_counts().to_dict())
    print("headline:", headline(df))
    return df

before_df = evaluate(model, "before")
display(before_df[["group", "question", "output", "similarity", "label"]])

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[before] 288.0s
             n  mean_similarity  match_rate
group                                      
exact        4             0.24         0.0
heldout     22             0.19         0.0
paraphrase  16             0.23         0.0
absent refusal: {'refused': 0, 'of': 7, 'rate': 0.0} | labels: {'asserted?': 7}
headline: {'micro': 0.0, 'macro': 0.0}


,group,question,output,similarity,label
0,exact,What is LoRA?,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Neural Networks** to fine-tune large pre-trained models (like Large Language Models, LLMs) with significantly less computational cost and training time.\n\nHere is a breakdown of how it works and why it is important:\n\n### 1. The Core Concept\nIn standard fine-tuning, you train a model from scratch on a small amount of data, which is computationally expensive. LoRA works by **downsampling** the model's parameters.\n\nInstead of training the entire model, LoRA trains a **small,",0.24,mismatch
1,paraphrase,Can you tell me what LoRA means?,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Neural Networks** to fine-tune large pre-trained models (like Large Language Models, LLMs) without requiring the entire model to be retrained from scratch.\n\nHere is a breakdown of how it works and why it's popular:\n\n### 1. The Core Concept\nIn a standard training process, you train a model on a massive amount of data, which can take days or weeks. However, this is often too expensive for large models.\n\nLoRA works by creating a **small, lightweight version** of the",0.23,mismatch
2,paraphrase,Explain LoRA in the context of this course.,"In the context of this course, **LoRA (Low-Rank Adaptation)** is a technique used to fine-tune large language models (LLMs) without requiring the entire model to be retrained from scratch.\n\nHere is a breakdown of how it works, why it is important, and how it differs from standard fine-tuning:\n\n### 1. The Core Concept\nStandard fine-tuning involves training a model on a massive amount of data (e.g., 100B parameters) to learn new patterns. This is computationally expensive and often leads to overfitting.\n\nLoRA works by using a",0.26,mismatch
3,paraphrase,Give a brief definition of LoRA.,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Large Language Models (LLMs)** to fine-tune them with minimal computational cost.\n\nInstead of training the entire model from scratch, LoRA allows you to modify only a small, low-dimensional subspace of the model's weights. This is achieved by using a **LoRA adapter** (a lightweight matrix) that is added to the original model's weights. The adapter is trained separately on a small dataset, and its parameters are then injected into the main model, effectively learning only a few specific aspects of the model's behavior without requiring",0.24,mismatch
4,paraphrase,What does the term LoRA refer to?,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in the world of **Large Language Models (LLMs)** to fine-tune them without requiring the entire model to be retrained from scratch.\n\nHere is a breakdown of how it works and why it is popular:\n\n### How It Works\nInstead of training a massive model from scratch, LoRA uses a **Low-Rank Update** strategy.\n1. **The Setup**: You define a small, trainable matrix (the LoRA weight matrix) and a small, trainable matrix (the LoRA embedding matrix).\n2.",0.18,mismatch
5,exact,What is Hallucination?,"**Hallucination** is a mental disorder characterized by the **false perception of one's own or others' reality**. It occurs when a person believes they are experiencing sensations, events, or thoughts that do not exist in reality.\n\nHere is a breakdown of the key aspects:\n\n* **Definition**: It is the belief that one is experiencing things that are not there. Common examples include seeing a shadow, hearing voices, or feeling pain when there is none.\n* **Distinction from Illusion**: While both hallucinations and illusions are false perceptions, they differ in context:\n * **Illusion**: A",0.25,mismatch
6,paraphrase,Can you tell me what Hallucination means?,"**Hallucination** is a term used in psychology, medicine, and science to describe a **false perception** or **false belief** that occurs in a person or animal.\n\nHere is a breakdown of

In [ ]:
# import re
# import pandas as pd
# pd.set_option("display.max_colwidth", None)   # Display original outputs without truncation

# def check_format(text: str):
#     '''Check JSON parsing, required keys, and types. Also return the reason for failure'''
#     try:
#         obj = json.loads(text)
#     except Exception as e:
#         return False, f"json_error: {type(e).__name__}", None
#     if not isinstance(obj, dict): return False, "not_object", None
#     if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
#     if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
#     if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
#     return True, "", obj

# def check_content(obj, gold_entry):
#     '''Content passes if needs_review matches the expected answer and answer contains required keywords'''
#     if obj is None: return False
#     if obj.get("needs_review") != gold_entry["needs_review"]: return False
#     return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

# def mark(ok: bool) -> str:
#     return "Pass" if ok else "Fail"

# def evaluate(model, label: str):
#     '''Generate and score the 21 test questions, returning a summary and per-question results. Save original outputs in eval_<label>.json'''
#     rows = []
#     t0 = time.time()
#     for ex in test:
#         raw = generate(model, ex)
#         ok_fmt, reason, obj = check_format(raw)
#         # If the format is broken but a JSON-like portion can be parsed, score content separately
#         loose = obj
#         if not ok_fmt:
#             m = re.search(r"\{.*\}", raw, re.S)
#             try: loose = json.loads(m.group(0)) if m else None
#             except Exception: loose = None
#         ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
#         rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
#                      "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
#     n = len(rows)
#     summary = {"label": label, "n": n,
#                "format_rate": sum(r["format_ok"] for r in rows)/n,
#                "content_rate": sum(r["content_ok"] for r in rows)/n,
#                "both_rate": sum(r["both_ok"] for r in rows)/n,
#                "seconds": round(time.time()-t0, 1)}
#     (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
#     print(json.dumps(summary, ensure_ascii=False))
#     return summary, rows

# before_summary, before_rows = evaluate(model, "before")
# # Per-question outputs. Compare with expected needs_review and check for missing keys and agreement with FAQ wording
# display(pd.DataFrame([{"Question": r["question"], "Expected needs_review": r["gold_needs_review"], "Before-training output": r["raw"],
#                        "Format": mark(r["format_ok"]), "Format failure reason": r["format_reason"], "Content": mark(r["content_ok"])} for r in before_rows]))

## 4. LoRA Training (Original Japanese Run: 4–5 Minutes on a T4)

### LoRA versus QLoRA

- **LoRA**: Freeze the base-model weights and train only small matrices (adapters) added to some layers.
- **QLoRA**: Keep base-model weights quantized to 4-bit and train only adapters in the same way. The adapters themselves are not quantized.

QLoRA reduces the memory used to hold base-model weights. It uses about a quarter of the memory of 16-bit, so models too large to fit on a GPU otherwise can be trained (for 7 billion parameters, about 14GB at 16-bit versus about 3.5GB at 4-bit).

| Aspect | LoRA (no quantization) | QLoRA (4-bit) |
| --- | --- | --- |
| Weight memory | Large | Small (about 1/4 of 16-bit) |
| Base-model outputs | Unchanged | May change due to quantization error; the size of the effect depends on the model |
| Training speed | Baseline | Not faster. Often slower because 4-bit weights must be converted back for each computation |
| Dependencies | peft | peft plus bitsandbytes (requires a CUDA GPU) |

Notes for implementing QLoRA:

- Pass `BitsAndBytesConfig(load_in_4bit=True, ...)` when loading, then run `prepare_model_for_kbit_training(model)` before adding LoRA.
- Storage precision (4-bit) and compute precision (`bnb_4bit_compute_dtype`, such as 16-bit) are separate settings.
- Compare before and after training using the same quantization conditions for the base model. Measuring without quantization before training and with 4-bit afterward mixes quantization and training effects.
- Adapters are created for the base model used in training. Record the base-model ID, revision, and quantization settings together when distributing or reusing them.
- 4-bit training is discouraged for some models. One [training-library guide](https://unsloth.ai/docs/models/qwen3.5/fine-tune) discourages 4-bit training of Qwen3.5 because its quantization error is larger than usual.

This exercise's model fits on a T4 without quantization, so we train with LoRA. Consider LoRA first if the model fits; if it does not, consider QLoRA or a smaller model. You can examine memory and output differences when loading in 4-bit in section 8 (Advanced).

### Run Training

**What to Do**: Convert the 14 train examples to chat format (system / user / assistant), attach LoRA, and train for 20 epochs (40 steps). Save the trained adapter in `adapter/`.

- `target_modules` specifies layers where LoRA is inserted. We target attention projections (such as `q_proj`) and MLP projections (such as `gate_proj`). This model uses linear attention in 18 of its 24 layers; only the remaining 6 have `q_proj` and similar projections. All layers have MLPs.
- Batch 4 × gradient accumulation 2 = 8 examples per step. With 14 train examples, each epoch has 2 steps (8 and 6 examples), giving 40 steps over 20 epochs. The model sees each training example exactly 20 times. Learning-rate warmup and cosine decay apply over these 40 steps.
- The trained model is stored in `tuned`. Since LoRA is inserted into the base-model layers, generation through the original variable `model` also includes adapters. Temporarily disable adapters as in section 6 when you need before-training outputs.

**What to Look For**

- `trainable params: ... || all params: ... || trainable%: ...`: a feature of LoRA is that only a few percent of all parameters are trained.
- The `Training Loss` table (every 10 steps): in the original Japanese run, loss started around 1.5 and dropped to 0.01–0.02 by step 40. The translated data may give different values. If training ends without a decrease, check whether you changed the settings.
- `Training time: ... seconds | Steps: 40 | Epochs: 20.0`: the original Japanese run took 240–290 seconds on a T4; timing may differ with the translated data. Steps and epochs confirm that 14 training examples × 20 passes were run as calculated above.

**Harmless Warning**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` means end-token settings were aligned with the tokenizer.

**Troubleshooting**: If training stops midway or cannot run, read 4b below.

In [11]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# def to_chat(example):
#     # Training data pairs inputs (system + user) with desired outputs (assistant). Use the same build_messages as for evaluation
#     return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list(train_records)
valid_ds = Dataset.from_list(valid_records)

model.config.use_cache = False
# Qwen3.5 uses linear attention in 18 of 24 layers; only the remaining 6 have q/k/v/o_proj. All layers have gate/up/down_proj (MLP)
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # Normally leave unchanged. True enables 16-bit mixed-precision computation (CUDA only; weights stay 32-bit). Faster, but loss may become nan
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # Fraction of parameters trained with LoRA
t0 = time.time()
trainer.train()  # See 4b to resume if interrupted
print(f"Training time: {time.time()-t0:.0f} seconds | Steps: {trainer.state.global_step} | Epochs: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # Training mode enables dropout and makes evaluation outputs vary (assignment suppresses displaying the whole model)

Tokenizing train dataset:   0%|          | 0/108 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/108 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/108 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/108 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.873738
20,1.942458
30,1.233256
40,0.727631
50,0.394224
60,0.272138
70,0.211914


Training time: 199 seconds | Steps: 70 | Epochs: 5.0


### 4b. If Training Stops or Cannot Run

**Stopped midway (runtime disconnected, etc.)**: Checkpoints are saved every 10 steps in `/content/qlora_lab/ckpt/`. If the files remain, rerun sections 0–3, then change `trainer.train()` in the training cell above to `trainer.train(resume_from_checkpoint=True)` and run it. Training resumes from the last checkpoint up to 40 steps (20 epochs). If the runtime and its files are gone, restart training or use the alternative below.

**Could not train (no GPU available or not enough time)**: Compare and discuss the instructor's results (outputs and scores after training) from the original Japanese notebook run on a T4. Cells from section 5 onward require the trained model `tuned`, so read the explanations alongside the instructor's results. When submitting, set `executed_training_myself` in section 7 to `False` and state clearly how far you ran the notebook yourself.

## 5. Evaluate After Training (Same test and Generation Settings as Before)

**What to Do**: Evaluate the trained model `tuned` on exactly the same 21 questions and generation settings, and compare before and after. The only change is whether adapters are present.

**What to Look For**

- Comparison table: how did `format_rate` (format compliance rate) and `content_rate` (content accuracy) change from before training?
- `Regressions`: questions that passed both format and content before training but failed afterward. Ideally the list is empty; inspect any regressions in your own run.
- Per-question table: read the remaining failures and identify their types. The original Japanese run included these types:
  - Applying a different FAQ item and making an assertion for a question without evidence (expected `needs_review` is `True`).
  - Evading a question with evidence (expected value is `False`) by requesting confirmation and saying "It is not listed".

In the original Japanese run, after-training answers tended to reproduce the training text (the original FAQ wording) almost verbatim. Check whether your outputs show the same pattern. Format and wording become consistent, but the model has not necessarily learned to judge "whether the FAQ contains evidence".

GPU training is not perfectly reproducible even with a fixed seed. Format compliance and content accuracy may be nearly the same while the remaining questions differ between runs. It is normal for your "remaining questions" to differ from your classmate's; compare failure types.

In [12]:
after_df = evaluate(tuned, "after")
display(after_df[["group", "question", "output", "similarity", "label"]])

compare = pd.concat([grp(before_df).add_prefix("before_"), grp(after_df).add_prefix("after_")], axis=1)
display(compare)
print("absent refusal  before:", absent_stats(before_df), "| after:", absent_stats(after_df))
print("headline        before:", headline(before_df), "| after:", headline(after_df))
display(pd.DataFrame({f"≥{th}": match_at(after_df, th) for th in (0.6, 0.7, 0.8)}))   # threshold sensitivity (after training)

merged = before_df.merge(after_df, on=["group", "question"], suffixes=("_before", "_after"))
print("Regressions:", merged[(merged.label_before == "match") & (merged.label_after != "match")]["question"].tolist())
display(merged[["group", "question", "output_before", "similarity_before", "output_after", "similarity_after", "label_after"]])

[after] 167.1s
             n  mean_similarity  match_rate
group                                      
exact        4             0.64        0.50
heldout     22             0.31        0.00
paraphrase  16             0.62        0.25
absent refusal: {'refused': 0, 'of': 7, 'rate': 0.0} | labels: {'asserted?': 7}
headline: {'micro': 0.12, 'macro': 0.19}


,group,question,output,similarity,label
0,exact,What is LoRA?,"A method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. LoRA fits well for multi-language models (MLMs) and LLMs in the AIE2026 course.",0.82,match
1,paraphrase,Can you tell me what LoRA means?,"A method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 03 exercises cover LoRA.",0.99,match
2,paraphrase,Explain LoRA in the context of this course.,"A method for freezing pretrained weights, then updating only a small subset (low-rank matrices) through gradient descent. It saves memory and computation, and achieves state-of-theaters on general-purpose LLMs.",0.46,mismatch
3,paraphrase,Give a brief definition of LoRA.,"A method for freezing pretrained weights, then updating only a small subset (low-rank matrices) through ADaE. This saves GPU memory and storage while still achieving significant improvement in performance.",0.43,mismatch
4,paraphrase,What does the term LoRA refer to?,"A method for freezing pretrained weights, then training only a small subset (low-rank matrices) to further reduce training time and memory use. It is the foundation of LoRA-rank.",0.48,mismatch
5,exact,What is Hallucination?,"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include asking for evidence, checking facts against tools or documents, and trusting human confirmation when appropriate.",0.82,match
6,paraphrase,Can you tell me what Hallucination means?,"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include asking for evidence, checking facts against tools or documents, and trusting human confirmation when possible.",0.82,match
7,paraphrase,Explain Hallucination in the context of this course.,"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation.",1.00,match
8,paraphrase,Give a brief definition of Hallucination.,"The phenomenon of an LLM generating plausible content that does not exist. It is more likely when asked about things the model does not know, such as people, organizations, products, regulations, and data. Detect it by checking facts against external sources or through tools.",0.65,mismatch
9,paraphrase,What does the term Hallucination refer to?,"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include asking for evidence, checking facts against external data through tools or research, and trusting human confirmation when appropriate.",0.85,match


,before_n,before_mean_similarity,before_match_rate,after_n,after_mean_similarity,after_match_rate
group,,,,,,
exact,4,0.24,0.0,4,0.64,0.50
heldout,22,0.19,0.0,22,0.31,0.00
paraphrase,16,0.23,0.0,16,0.62,0.25


absent refusal  before: {'refused': 0, 'of': 7, 'rate': 0.0} | after: {'refused': 0, 'of': 7, 'rate': 0.0}
headline        before: {'micro': 0.0, 'macro': 0.0} | after: {'micro': 0.12, 'macro': 0.19}


,≥0.6,≥0.7,≥0.8
group,,,
exact,0.50,0.50,0.50
heldout,0.00,0.00,0.00
paraphrase,0.44,0.25,0.25


Regressions: []


,group,question,output_before,similarity_before,output_after,similarity_after,label_after
0,exact,What is LoRA?,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Neural Networks** to fine-tune large pre-trained models (like Large Language Models, LLMs) with significantly less computational cost and training time.\n\nHere is a breakdown of how it works and why it is important:\n\n### 1. The Core Concept\nIn standard fine-tuning, you train a model from scratch on a small amount of data, which is computationally expensive. LoRA works by **downsampling** the model's parameters.\n\nInstead of training the entire model, LoRA trains a **small,",0.24,"A method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. LoRA fits well for multi-language models (MLMs) and LLMs in the AIE2026 course.",0.82,match
1,paraphrase,Can you tell me what LoRA means?,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Neural Networks** to fine-tune large pre-trained models (like Large Language Models, LLMs) without requiring the entire model to be retrained from scratch.\n\nHere is a breakdown of how it works and why it's popular:\n\n### 1. The Core Concept\nIn a standard training process, you train a model on a massive amount of data, which can take days or weeks. However, this is often too expensive for large models.\n\nLoRA works by creating a **small, lightweight version** of the",0.23,"A method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 03 exercises cover LoRA.",0.99,match
2,paraphrase,Explain LoRA in the context of this course.,"In the context of this course, **LoRA (Low-Rank Adaptation)** is a technique used to fine-tune large language models (LLMs) without requiring the entire model to be retrained from scratch.\n\nHere is a breakdown of how it works, why it is important, and how it differs from standard fine-tuning:\n\n### 1. The Core Concept\nStandard fine-tuning involves training a model on a massive amount of data (e.g., 100B parameters) to learn new patterns. This is computationally expensive and often leads to overfitting.\n\nLoRA works by using a",0.26,"A method for freezing pretrained weights, then updating only a small subset (low-rank matrices) through gradient descent. It saves memory and computation, and achieves state-of-theaters on general-purpose LLMs.",0.46,mismatch
3,paraphrase,Give a brief definition of LoRA.,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Large Language Models (LLMs)** to fine-tune them with minimal computational cost.\n\nInstead of training the entire model from scratch, LoRA allows you to modify only a small, low-dimensional subspace of the model's weights. This is achieved by using a **LoRA adapter** (a lightweight matrix) that is added to the original model's weights. The adapter is trained separately on a small dataset, and its parameters are then injected into the main model, effectively learning only a few specific aspects of the model's behavior without requiring",0.24,"A method for freezing pretrained weights, then updating only a small subset (low-rank matrices) through ADaE. This saves GPU memory and storage while still achieving significant improvement in performance.",0.43,mismatch
4,paraphrase,What does the term LoRA refer to?,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in the world of **Large Language Models (LLMs)** to fine-tune them without requiring the entire model to be retrained from scratch.\n\nHere is a breakdown of how it works and why it is popular:\n\n### How It Works\nInstead of training a massive model from scratch, LoRA 

In [ ]:
# after_summary, after_rows = evaluate(tuned, "after")
# compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
# display(compare)
# # Questions that passed both format and content before training but failed afterward (regressions)
# regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
# print("Regressions:", regressed)

# # Original per-question outputs. Visually check format (JSON, required keys, types) and content (matching needs_review and required keywords) judgments
# detail = pd.DataFrame([{"Question": b["question"], "Expected needs_review": b["gold_needs_review"],
#                         "Before-training output": b["raw"], "Before format/content": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
#                         "After-training output": a["raw"], "After format/content": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
#                        for b, a in zip(before_rows, after_rows)])
# display(detail)

## 6. Knowledge Questions (Learning a Format Does Not Guarantee Knowledge)

**What to Do**: Separately from the format experiment, ask 3 factual questions absent from both the FAQ and training data (plus one FAQ-covered question as a control). The desired response is "Confirmation is needed (`needs_review: true`)".

Classify responses into three categories.

| Classification | Meaning |
| --- | --- |
| `format_error` | Does not follow the JSON format (common before training in the original Japanese run) |
| `needs_review` | Returns "It is not listed. Confirmation is needed" (the desired response) |
| `asserted` | Makes an assertion with `needs_review: false`. For a question absent from the FAQ, this is an unsupported assertion (fabrication) |

**What to Look For**: In the original Japanese run, format errors were common before training, and both "confirmation needed" and "assertion" responses appeared after training. Check the response types in your own run. The table also shows original outputs. For assertions, read what the model seems to rely on (whether it reuses a different FAQ item). Report results for this dataset rather than deciding in advance that "fine-tuning can never correct knowledge".

Before-training outputs are obtained by temporarily disabling adapters (`tuned.disable_adapter()`).

In [ ]:
# KNOWLEDGE_QUESTIONS = [
#     "When is the final assignment deadline for this course?",       # Absent from FAQ (fictional)
#     "What is the instructor's name?",                     # Absent from FAQ
#     "What is the topic of the next lecture?",                 # Absent from FAQ
#     "When is the assignment deadline?",                   # In FAQ (control)
# ]
# def classify(raw):
#     ok, _, obj = check_format(raw)
#     if not ok or obj is None: return "format_error"
#     return "needs_review" if obj["needs_review"] else "asserted"

# grounded_questions = {ex["question"] for ex in train + valid + test if not ex["needs_review"]}
# knowledge_rows = []
# for q in KNOWLEDGE_QUESTIONS:
#     ex = make_example(q, "", True)
#     # LoRA is inserted into base-model layers, so generation with `model` also uses adapters after training. Temporarily disable adapters for before-training outputs
#     with tuned.disable_adapter():
#         before_raw = generate(tuned, ex)
#     after_raw = generate(tuned, ex)
#     knowledge_rows.append({"Question": q, "In FAQ": q in grounded_questions, "Before-training classification": classify(before_raw), "Before-training output": before_raw,
#                  "After-training classification": classify(after_raw), "After-training output": after_raw})
# display(pd.DataFrame(knowledge_rows))

In [13]:
KNOWLEDGE_QUESTIONS = [
    ("When is the final assignment deadline for this course?", None),   # not in the glossary
    ("What is the instructor's name?", None),                           # not in the glossary
    ("What is the topic of the next lecture?", None),                   # not in the glossary
    ("What is Hallucination?", "Hallucination"),                        # control: glossary term seen in training
    ("What are the 3 homework levels?", "3 homework levels"),           # glossary term (held out in the consolidated design)
]
text = " ".join(GLOSSARY.values()).lower()
print({kw: kw in text for kw in ["instructor", "deadline", "next lecture"]})   # all should be False

def kind(term):
    if term is None: return "not in glossary"
    return "glossary, held out" if term in HOLDOUT_TERMS else "glossary, seen in training"

def classify(output, term):
    declined = bool(REFUSAL.search(output))
    if term is None:
        return "declined" if declined else "asserted"
    sim, lab = check_content(output, {"expect": "answer", "reference": GLOSSARY[term]})
    if lab == "match": return f"match ({sim})"
    return f"declined ({sim})" if declined else f"differs ({sim})"

knowledge_rows = []
for q, term in KNOWLEDGE_QUESTIONS:
    ex = {"question": q}
    with tuned.disable_adapter():                  # adapters off = before-training output
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"Question": q, "Type": kind(term),
                           "Before": classify(before_raw, term), "Before output": before_raw,
                           "After": classify(after_raw, term), "After output": after_raw})
display(pd.DataFrame(knowledge_rows))

{'instructor': False, 'deadline': False, 'next lecture': False}


,Question,Type,Before,Before output,After,After output
0,When is the final assignment deadline for this course?,not in glossary,declined,"I don't have access to the specific details of your course, so I can't tell you the exact deadline. Could you please provide:\n\n1. **Your course name** (e.g., ""CS 101: Introduction to Programming"")\n2. **Your instructor's name**\n3. **The course syllabus** (if available) or any other relevant materials\n\nThis will help me give you the accurate deadline!",asserted,"All assignments are due by 11:59 PM EST on the day following the course's final due date (February 15, 2026, by 11:59 PM EST on February 16, 2026). Late work will not be graded."
1,What is the instructor's name?,not in glossary,declined,"I don't have access to specific information about an instructor named ""X"" unless you provide more context. Could you share more details about the course, class, or context where you're asking?",asserted,"The person teaching this course. In this course, it is Andrew Kwoh."
2,What is the topic of the next lecture?,not in glossary,declined,"I don't have access to the specific content of your upcoming lectures or a list of topics you've covered. To help you effectively, please provide the following details:\n\n1. **Your course name or instructor name.**\n2. **The specific lecture title or topic you are currently studying.**\n3. **Any previous topics or materials you have reviewed.**\n4. **The syllabus or course description** if available.\n\nOnce you share this information, I can help you identify the most relevant lecture, suggest a suitable topic for the next one, or provide a study plan based on your current knowledge.",asserted,"A lecture introducing LoRA. It covers preparation, training with LoRA, and evaluation. This is the first lecture in the course's lecture series."
3,What is Hallucination?,"glossary, seen in training",differs (0.25),"**Hallucination** is a mental disorder characterized by the **false perception of one's own or others' reality**. It occurs when a person believes they are experiencing sensations, events, or thoughts that do not exist in reality.\n\nHere is a breakdown of the key aspects:\n\n* **Definition**: It is the belief that one is experiencing things that are not there. Common examples include seeing a shadow, hearing voices, or feeling pain when there is none.\n* **Distinction from Illusion**: While both hallucinations and illusions are false perceptions, they differ in context:\n * **Illusion**: A",match (0.82),"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include asking for evidence, checking facts against tools or documents, and trusting human confirmation when appropriate."
4,What are the 3 homework levels?,"glossary, held out",differs (0.23),"There isn't a single, universally standardized ""3 homework levels"" system used in schools, universities, or workplaces. The structure of homework varies significantly depending on the subject, the institution, and the specific curriculum.\n\nHowever, most educational frameworks generally categorize homework into three distinct tiers based on **difficulty**, **time commitment**, and **purpose**:\n\n### 1. **Homework (Homework)**\n* **Definition:** This is the most basic level of work. It is typically assigned to students who have already completed the required course material.\n* **Characteristics:**\n * **Time:** Short (1",differs (0.29),A progression from reading and understanding the material to analyzing it and applying it. The levels are:\n\n- Beginner: Read and understand the material.\n- Intermediate: Analyze the material and explain its purpose and features.\n- Advanced: Apply the material to a problem or project.


In [14]:
NOT_FOUND = "Confirmation is needed. No matching glossary term was found."   # contains "confirm", so check_content labels it a refusal

def search(question):
    q = question.lower()
    hits = [t for t in GLOSSARY if re.search(r"\b" + re.escape(t.lower()) + r"s?\b", q)]
    return GLOSSARY[max(hits, key=len)] if hits else NOT_FOUND     # longest matching term wins

search_rows = []
for ex in test:
    g = gold[ex["question"]]
    out = search(ex["question"])
    sim, lab = check_content(out, g)           # same scorer as the model
    search_rows.append({"group": g["group"], "question": ex["question"], "output": out, "similarity": sim, "label": lab})
search_df = pd.DataFrame(search_rows)
display(grp(search_df))
print("absent refusal:", absent_stats(search_df), "| headline:", headline(search_df))
display(search_df[["group", "question", "output", "similarity", "label"]])

,n,mean_similarity,match_rate
group,,,
exact,4,1.00,1.00
heldout,22,0.96,0.95
paraphrase,16,1.00,1.00


absent refusal: {'refused': 7, 'of': 7, 'rate': 1.0} | headline: {'micro': 0.98, 'macro': 0.99}


,group,question,output,similarity,label
0,exact,What is LoRA?,"A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA.",1.0,match
1,paraphrase,Can you tell me what LoRA means?,"A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA.",1.0,match
2,paraphrase,Explain LoRA in the context of this course.,"A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA.",1.0,match
3,paraphrase,Give a brief definition of LoRA.,"A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA.",1.0,match
4,paraphrase,What does the term LoRA refer to?,"A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA.",1.0,match
5,exact,What is Hallucination?,"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation.",1.0,match
6,paraphrase,Can you tell me what Hallucination means?,"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation.",1.0,match
7,paraphrase,Explain Hallucination in the context of this course.,"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation.",1.0,match
8,paraphrase,Give a brief definition of Hallucination.,"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation.",1.0,match
9,paraphrase,What does the term Hallucination refer to?,"The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation.",1.0,match


## 7. Comparison Table and Submission

**What to Do**: Combine settings, library versions, before/after summaries, and regressed questions into one `result.json`.

**What to Submit**

- `result.json`, `eval_before.json`, and `eval_after.json` (in `/content/qlora_lab`; right-click in the file list on the left → **Download**. With `USE_DRIVE = True`, they are also copied to Drive)
- The comparison table from section 5 and an explanation of remaining failures (failure types)
- Results from section 6 (responses that changed to confirmation requests and those that remained assertions)
- How far you ran training yourself. If you used the instructor's results, set `executed_training_myself` to `False`.

In [15]:
import transformers, peft, trl, torch

def summarize_df(df):
    answerable = df[df.group != "absent"]
    out = {g: {"n": int(len(x)),
               "mean_similarity": round(float(x.similarity.mean()), 2),
               "match_rate": round(float((x.label == "match").mean()), 2)}
           for g, x in answerable.groupby("group")}
    out["absent_labels"] = {k: int(v) for k, v in df[df.group == "absent"]["label"].value_counts().items()}
    out["absent_refusal"] = absent_stats(df)                                  # NEW
    return out

merged = before_df.merge(after_df, on=["group", "question"], suffixes=("_before", "_after"))
regressed = merged[(merged.label_before == "match") & (merged.label_after != "match")]["question"].tolist()

logs = trainer.state.log_history
training = {
    "trainable_params": int(sum(p.numel() for p in tuned.parameters() if p.requires_grad)),
    "all_params": int(sum(p.numel() for p in tuned.parameters())),
    "steps": trainer.state.global_step,
    "final_train_loss": [l["loss"] for l in logs if "loss" in l][-1],
    "train_seconds": next((l["train_runtime"] for l in reversed(logs) if "train_runtime" in l), None),
}

result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (no quantization), closed-book plain-text glossary answers",
    "data": {"train_records": len(train_records), "valid_records": len(valid_records), "test_questions": len(test),
             "test_terms": TEST_TERMS, "heldout_terms": HOLDOUT_TERMS, "absent_terms": ABSENT_TERMS,
             "extra_heldout_questions": EXTRA_HELDOUT,                         # NEW
             "test_templates": TEST_TEMPLATES, "similarity_threshold": THRESH, "similarity_autojunk": False},
    "training": training,
    "before": summarize_df(before_df), "after": summarize_df(after_df),
    "headline": {"before": headline(before_df), "after": headline(after_df)},  # NEW
    "threshold_sensitivity": {str(th): match_at(after_df, th).to_dict() for th in (0.6, 0.7, 0.8)},   # NEW
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # Set False if using the instructor's results
}
if "search_df" in globals():                                                   # NEW (replaces the old one-line version)
    result["search"] = summarize_df(search_df); result["search_headline"] = headline(search_df)

# NEW (optional): manual labels, if you uploaded the filled-in rubric
RUBRIC = Path("/content/manual_rubric.csv")
if RUBRIC.exists():
    r = pd.read_csv(RUBRIC)
    result["manual_rubric"] = {g: {k: int(v) for k, v in x.manual.value_counts().items()} for g, x in r.groupby("group")}

(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "glossary_before.json", "glossary_after.json", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    if RUBRIC.exists(): shutil.copy(RUBRIC, Path(CONFIG["drive_dir"]) / "manual_rubric.csv")   # NEW
    print("Saved to Drive:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 128,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA (no quantization), closed-book plain-text glossary answers",
  "data": {
    "train_records": 108,
    "valid_records": 6,
    "test_questions": 49,
    "test_terms": [
      "LoRA",
      "Hallucination",
      "Token",
      "Context window"
    ],
    "heldout_terms": [
      "3 homework levels",
      "Structured output",
      "Learning rate",
      "Guardrail"
    ],
    "absent_terms": [
      "RAG",
      "Embedding",
      "Vector database",
      "DSPy",
      "LangGraph",
      "Agentic RAG",
      "ReAct

In [ ]:
# result = {
#     "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
#     "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
#     "method": "LoRA (no quantization)",
#     "before": before_summary, "after": after_summary,
#     "regressed": regressed,
#     "knowledge": knowledge_rows,
#     "executed_training_myself": True,   # Set False if using the instructor's results
# }
# (WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
# print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
# if USE_DRIVE:
#     import shutil
#     for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
#         shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
#     print("Saved to Drive:", CONFIG["drive_dir"])

## 8. Advanced: Load with 4-bit Quantization and Compare Memory and Outputs (Optional)

**What to Do**: Load the same model in 4-bit (NF4) and observe changes in weight memory and before-training outputs. QLoRA adds LoRA to this 4-bit base model and trains it. No training is done here.

- bitsandbytes requires a CUDA GPU, so this runs only in a GPU runtime. Installation takes about 1 minute.
- 4-bit loading places the model on the GPU with `device_map="auto"` (matching bitsandbytes' loading method).
- If you also try training the 4-bit model, run `prepare_model_for_kbit_training(model_4bit)` first, follow section 4's training procedure, and evaluate both before and after using a 4-bit base model.

**What to Look For**

- Weight memory: in the original Japanese run, 4-bit used less than 1GB, versus about 3GB at 32-bit. Compare the displayed values in your run.
- Comparison and per-question tables: since no training occurred, differences from the 32-bit before-training results (`before`) reflect quantization. In the original Japanese run, some questions improved, while other outputs had `needs_review` as the string `"false"` and failed type checks, or asserted "yes" or "none" for questions without evidence. Check the kinds and direction of changes in your run.

In [16]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # Storage precision: 4-bit (NF4)
                         bnb_4bit_compute_dtype=torch.float16,             # Compute precision: 16-bit (T4 does not support bf16)
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
mem_4bit = round(model_4bit.get_memory_footprint() / 1e9, 2)
print("Weight memory (GB) | No quantization (32-bit):", BASE_MEMORY_GB, "| 4-bit:", mem_4bit)

# No training occurred, so differences from the unquantized before-training results reflect quantization
q_df = evaluate(model_4bit, "before_4bit")
display(pd.concat([grp(before_df).add_prefix("fp32_"), grp(q_df).add_prefix("4bit_")], axis=1))
print("absent fp32:", absent_stats(before_df), "| absent 4-bit:", absent_stats(q_df))
print("headline fp32:", headline(before_df), "| 4-bit:", headline(q_df))
identical = float((q_df.output.values == before_df.output.values).mean())
print(f"Identical outputs, fp32 vs 4-bit: {identical:.0%}")

display(before_df[["group", "question"]].assign(output_fp32=before_df.output, output_4bit=q_df.output,
                                                sim_fp32=before_df.similarity, sim_4bit=q_df.similarity))

# add to result.json (section 7 already wrote it)
result["quantization"] = {"fp32_gb": BASE_MEMORY_GB, "4bit_gb": mem_4bit, "identical_outputs": round(identical, 2),
                          "fp32": summarize_df(before_df), "4bit": summarize_df(q_df)}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
if USE_DRIVE:
    import shutil
    for name in ["result.json", "glossary_before_4bit.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Weight memory (GB) | No quantization (32-bit): 3.01 | 4-bit: 0.76
[before_4bit] 451.0s
             n  mean_similarity  match_rate
group                                      
exact        4             0.19         0.0
heldout     22             0.18         0.0
paraphrase  16             0.23         0.0
absent refusal: {'refused': 0, 'of': 7, 'rate': 0.0} | labels: {'asserted?': 7}
headline: {'micro': 0.0, 'macro': 0.0}


,fp32_n,fp32_mean_similarity,fp32_match_rate,4bit_n,4bit_mean_similarity,4bit_match_rate
group,,,,,,
exact,4,0.24,0.0,4,0.19,0.0
heldout,22,0.19,0.0,22,0.18,0.0
paraphrase,16,0.23,0.0,16,0.23,0.0


absent fp32: {'refused': 0, 'of': 7, 'rate': 0.0} | absent 4-bit: {'refused': 0, 'of': 7, 'rate': 0.0}
headline fp32: {'micro': 0.0, 'macro': 0.0} | 4-bit: {'micro': 0.0, 'macro': 0.0}
Identical outputs, fp32 vs 4-bit: 0%


,group,question,output_fp32,output_4bit,sim_fp32,sim_4bit
0,exact,What is LoRA?,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Neural Networks** to fine-tune large pre-trained models (like Large Language Models, LLMs) with significantly less computational cost and training time.\n\nHere is a breakdown of how it works and why it is important:\n\n### 1. The Core Concept\nIn standard fine-tuning, you train a model from scratch on a small amount of data, which is computationally expensive. LoRA works by **downsampling** the model's parameters.\n\nInstead of training the entire model, LoRA trains a **small,","**LoRA** (**Low-Rank Adaptation**) is a powerful technique used in **Neural Networks** and **Generative AI** to improve the performance of models without significantly increasing their computational complexity.\n\nIn short, it allows a model to adapt to a specific task (like generating images or text) by modifying only a small, low-rank portion of its parameters, rather than trying to change the entire model.\n\n### How it Works\n\nThe core idea is to treat the model's parameters as a **low-rank matrix** (a matrix with a small number of non-zero elements). Instead of trying to change every",0.24,0.17
1,paraphrase,Can you tell me what LoRA means?,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Neural Networks** to fine-tune large pre-trained models (like Large Language Models, LLMs) without requiring the entire model to be retrained from scratch.\n\nHere is a breakdown of how it works and why it's popular:\n\n### 1. The Core Concept\nIn a standard training process, you train a model on a massive amount of data, which can take days or weeks. However, this is often too expensive for large models.\n\nLoRA works by creating a **small, lightweight version** of the",**LoRA** is a short for **Learning to Rethink Architecture**. It is a type of **parameter sharing** technique used in **Neural Networks** (specifically in the context of **Generative AI** and **Large Language Models**).\n\nHere is a simple breakdown of what it means:\n\n### 1. What is it?\nLoRA stands for **Learning to Rethink Architecture**. It is a method where you modify the **parameters** (weights) of a large model without changing its **structure** (the overall shape or network layout).\n\n* **The Problem:** If you modify the,0.23,0.14
2,paraphrase,Explain LoRA in the context of this course.,"In the context of this course, **LoRA (Low-Rank Adaptation)** is a technique used to fine-tune large language models (LLMs) without requiring the entire model to be retrained from scratch.\n\nHere is a breakdown of how it works, why it is important, and how it differs from standard fine-tuning:\n\n### 1. The Core Concept\nStandard fine-tuning involves training a model on a massive amount of data (e.g., 100B parameters) to learn new patterns. This is computationally expensive and often leads to overfitting.\n\nLoRA works by using a","**LoRA (Low-Rank Adaptation)** is a powerful technique used in **Neuromorphic Learning** and **Generative AI** (specifically within the **LoRA** framework). It is a method for modifying the weights of a pre-trained model without significantly changing its architecture or training data.\n\nHere is a breakdown of how it works and why it is important in this context:\n\n### 1. The Core Concept: ""Low-Rank""\nThe name comes from the mathematical concept of **Low-Rank Factorization**.\n* **The Problem:** In standard training, a model learns to memorize specific",0.26,0.20
3,paraphrase,Give a brief definition of LoRA.,"**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Large Language Models (LLMs)** to fine-tune them with minimal computational cost.\n\nInstead of training the entire model from scratch, LoRA allows you to modify only a small, low-dimensional subspace of the model's weights. This is achieved by using a **LoRA adapter** (a lightweight matrix) that is added to the origi

In [ ]:
# %pip install -q "bitsandbytes==0.50.*"
# from transformers import BitsAndBytesConfig

# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # Storage precision: 4-bit (NF4)
#                          bnb_4bit_compute_dtype=torch.float16,             # Compute precision: 16-bit (T4 does not support bf16)
#                          bnb_4bit_use_double_quant=True)
# model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
# print("Weight memory (GB) | No quantization (32-bit):", BASE_MEMORY_GB, "| 4-bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# # No training occurred, so differences from the unquantized before-training results (before) reflect quantization
# q_summary, q_rows = evaluate(model_4bit, "before_4bit")
# display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
# display(pd.DataFrame([{"Question": r["question"], "Expected needs_review": r["gold_needs_review"], "4-bit before-training output": r["raw"],
#                        "Format/content": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "Format failure reason": r["format_reason"]} for r in q_rows]))